In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import time
import tqdm
import helpers

import environments
import environments.bicycle as bicycle

import controllers.clothoids as clothoids
import controllers.purepursuit as purepursuit

import metrics

import numpy as np
import gymnasium as gym
import matplotlib.pyplot as plt

/Users/nadir/Documents/research-project/.venv.mac/lib/python3.12/site-packages/pygame/pkgdata.py:25: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import resource_stream, resource_exists


In [4]:
env = environments.bicycle.BicycleCarEnv(
    road_network=bicycle.RoadNetwork(roads=[
        bicycle.create_rectangular_track(
            center=(50.0, 50.0),
            length=80.0,
            width=40.0,
            turn_radius=8.0,
            lane_config=bicycle.lane_config_from_width(8.0, num_lanes=1),
        )
    ]),
    # road_network=bicycle.create_rectangular_track_with_cross(
    #     center=(50.0, 50.0),
    #     length=80.0,
    #     width=40.0,
    #     cross_width=4.5,
    #     turn_radius=8.0,
    #     lane_config=bicycle.lane_config_from_width(8.0, num_lanes=1),
    # ),
    render_mode="rgb_array",
    spawn=((50.0, 30.0), 0.0),
    goal=((10.0, 50.0), 2.0),
    obstacles=[
        bicycle.Circle(center=(90, 50), radius=1.0),
    ],
    solid_road_borders=True,
    # 0.1second = 100ms per step
    dt=0.1
)

In [5]:
env.world_size

array([100, 100])

In [ ]:
controller = clothoids.ClothoidTentaclesController(
    num_tentacles=41,
    # t0=7.0,
    t0=10.0,
    # l0=5.0,
    l0=10.0,
    num_points_per_tentacle=64,

    wheelbase=env.WHEELBASE,
    vehicle_width=env.CAR_WIDTH,
    max_lateral_acceleration=4.0,
    max_deceleration=1.5,

    # clearance, curvature, trajectory
    weights=(0.1, 0.2, 0.5),

    # TODO: I set a target velocity of 3.5 but the controller always goes at 5.0 m/s
    target_velocity=3.5,
    kp_velocity=2.0,
)

In [ ]:
obs, info = env.reset()

for i in range(500):
    action = controller.get_action(
        observation=obs,
        path=env.global_path,
        obstacles=env.obstacles,
        road_network=env.road_network,
        max_steering=env.MAX_STEERING,
        max_acceleration=env.MAX_ACCELERATION,
    )
    
    obs, reward, terminated, truncated, info = env.step(action)
    
    env.overlay_manager.clear()
    controller.draw_debug(env, obs, env.global_path)
    
    helpers.preview(env)

    if terminated or truncated:
        break

env.close()

In [ ]:
# Get episode data and compute metrics
episode_data = env.get_episode_data()
# print(f"Episode finished after {i+1} steps")

# Compute metrics
print("\n=== Performance Metrics ===")

cte_metrics = metrics.compute_cross_track_error(
    positions=episode_data['positions'],
    reference_path=env.global_path,
)
print(f"CTE RMS: {cte_metrics['cte_rms']:.3f} m")

smoothness_metrics = metrics.compute_steering_smoothness(
    steering_angles=episode_data['steering_angles'],
    dt=env.dt,
)
print(f"Steering Jerk RMS: {smoothness_metrics['steering_jerk_rms']:.3f} rad/s³")

success = metrics.compute_success_rate([episode_data])
print(f"Success: {'Yes' if success == 1.0 else 'No'}")